# _load_data - rebuild Delta tables from bundled parquet

In [ ]:
WS = "163ba38c-3869-406f-adb7-37cbc981390c"   # rebound to target workspace id
LH = "7e08480c-cf8d-4206-901d-38b74dbe35d9"   # rebound to target lakehouse id
BASE   = f"abfss://{WS}@onelake.dfs.fabric.microsoft.com/{LH}"
IMPORT = f"{BASE}/Files/solution_import/lakehouse"
TABLES = f"{BASE}/Tables"

import json
try:
    import notebookutils; fs = notebookutils.fs
except Exception:
    from notebookutils import mssparkutils; fs = mssparkutils.fs

manifest = json.loads(fs.head(f"{IMPORT}/_manifest.json", 4 * 1024 * 1024))
print(f"{len(manifest)} tables to load")

loaded = 0
for m in manifest:
    sch, tbl = m["schema"], m["table"]
    src = f"{IMPORT}/{sch}/{tbl}"
    dst = f"{TABLES}/{sch}/{tbl}"
    try:
        df = spark.read.parquet(src)
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(dst)
        loaded += 1
        print(f"loaded {sch}.{tbl}: {df.count():,} rows")
    except Exception as e:
        print(f"WARN {sch}.{tbl}: {repr(e)[:160]}")

print(f"Done. Loaded {loaded}/{len(manifest)} tables.")